# Lab 11 — Student Version (Google Colab)
## Deploy a Fine-tuned Sentiment Classifier as a Gradio App

This is a **student-learning version** of Lab 11. It keeps **every topic and step** from the lab and adds short explanations of **what you are learning** and **why each step matters**.

## Lab overview
You will:
1. **Fine-tune** a compact transformer sentiment model (DistilBERT) on a small IMDb subset.
2. **Save** the trained model + tokenizer for reuse.
3. **Deploy** the model inside an interactive **Gradio web app**.
4. **Test** the deployed app with positive, negative, and mixed reviews.

### Estimated completion time
- **30 minutes**

### Runtime type (important)
- Set Runtime type to: **T4 GPU**


# Task 1: Fine-tune a sentiment model and save it

### What you are learning
- Fine-tune a pretrained transformer for **binary classification**.
- Build a tokenized **PyTorch Dataset** for text.
- Use **AMP** + **gradient clipping** for faster/stabler GPU training.
- Save model + tokenizer for deployment.

---

## Step 1: Install dependencies (run once)


In [ ]:
!pip install -q transformers datasets


## Step 2: Switch runtime to T4 GPU (required)
Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

## Step 3: Restart the session (required)
Runtime → Restart session

After restarting, re-run the notebook from the top.


## Step 4: Import required libraries


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm


## Step 5–6: Upload the IMDb dataset (IMDB Dataset.csv)


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


## Step 7: Read, shuffle, label-encode, and limit to 1000 rows


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
df = df[["review", "label"]]
df = df[:1000]  # For speed

print("Rows used:", len(df))
print(df.head(3))
print("Label counts:")
print(df["label"].value_counts())


## Step 7 continued: Tokenize and wrap in a Dataset class


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=256)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["review"], df["label"], test_size=0.2, random_state=42
)

train_dataset = IMDBDataset(train_texts.tolist(), train_labels.tolist())
val_dataset   = IMDBDataset(val_texts.tolist(), val_labels.tolist())

print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))


## Step 8: Initialize model + optimizer + DataLoaders


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

print("Using device:", device)


## Step 8 continued: Training loop with AMP + gradient clipping


In [ ]:
# AMP imports: support both torch.amp and torch.cuda.amp
try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda")
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        if device.type == "cuda":
            with autocast("cuda"):
                outputs = model(**batch)
                loss = outputs.loss
        else:
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


## Step 9: Save the trained model and tokenizer


In [ ]:
model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")
print("Saved model + tokenizer to ./sentiment_model")


# Task 2: Building a Gradio web app for sentiment analysis

### What you are learning
- Load your saved model for inference.
- Wrap the predictor in a simple **Gradio UI**.
- Launch with a shareable link.

---

## Step 1: Install Gradio


In [ ]:
!pip install -q gradio transformers


## Step 2: Import + load the fine-tuned model/tokenizer


In [ ]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import gradio as gr

tokenizer = DistilBertTokenizerFast.from_pretrained("sentiment_model")
model = DistilBertForSequenceClassification.from_pretrained("sentiment_model")
model.eval()

print("Loaded fine-tuned model from sentiment_model")


## Step 3: Define predict_sentiment(text) → Positive/Negative


In [ ]:
def predict_sentiment(text: str) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    # move to same device as model (works on CPU or GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        prediction = torch.argmax(logits, dim=1).item()

    return "Positive" if prediction == 1 else "Negative"


## Step 4: Create the Gradio interface


In [ ]:
demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=4, label="Enter a movie review"),
    outputs=gr.Textbox(label="Predicted Sentiment"),
    title="Fine-Tuned Sentiment Classifier",
    description="Type a movie review and see whether the model predicts it as Positive or Negative."
)


## Step 5: Launch Gradio (share=True)


In [ ]:
demo.launch(share=True)


# Task 3: Testing the deployed Gradio sentiment app

### What you are learning
Test the function directly first, then test via the Gradio web UI link.

---

## Step 1: Verify predict_sentiment with sample inputs


In [ ]:
print("Test 1:", predict_sentiment("This movie was absolutely fantastic! I loved every second of it."))
print("Test 2:", predict_sentiment("This was the worst movie I have ever seen. Totally boring and a waste of time."))
print("Test 3:", predict_sentiment("The movie was okay, not great but not terrible either."))


## Step 2: Interact through the Gradio interface
- Scroll up to the output of the `demo.launch(share=True)` cell.
- Click the public URL (it looks like `https://xxxxxxxxxx.gradio.live`).
- A new browser tab will open with the Gradio app.

## Step 3: Test a clearly positive review
Type:
> This movie was amazing! The story, acting, and visuals were all outstanding.
Click **Submit**.

## Step 5: Test a clearly negative review
Type:
> I really disliked this movie. The plot was weak and the acting was terrible.
Click **Submit**.

## Step 7: Experiment with your own sentences
Try neutral/mixed opinions and see how the model behaves.


# Lab review

1. What is the primary purpose of using Gradio in this lab?

A. To train the model on a GPU  
B. To create an interactive web interface for the model  
C. To package the model into a Docker container  
D. To deploy the model to Hugging Face Hub  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answer</strong></summary>

**B**

</details>
